# 01 · Training and evaluation of the Dynamic ViT

Code for *"Evaluation of a Dynamic Vision Transformer for Petrographic Fabric
Classification of Levantine Archaeological Ceramics"*.

This notebook trains the DynamicViT (LV-ViT-M backbone with token pruning) with
knowledge distillation and evaluates it. It covers three experiments, selected
in the configuration cell:

| `EXPERIMENT` | Description | Paper |
|---|---|---|
| `"main"` | 70/15/15 split of the dataset | Table 3, Fig. 2 |
| `"heldout"` | test on unseen samples per fabric | Table 4, Fig. 8 |
| `"sites"` | test on held-out archaeological sites | Suppl. Table S2, Fig. S21 |

Each experiment was run with seeds **5, 16 and 25** (each seed defines both the
data split and the initialisation); results are reported as mean ± std.
The figures in the paper refer to seed 5.

A GPU runtime is required (in Colab: *Runtime → Change runtime type → GPU*).

## 1. Setup

In [ ]:
# Dependencies (timm 0.6.13 is required by the DynamicViT code)
!pip install -q tensorboardX gdown timm==0.6.13

# Official DynamicViT implementation (Rao et al., NeurIPS 2021, MIT License)
!git clone -q https://github.com/raoyongming/DynamicViT.git

# Compatibility fix: torch._six was removed in recent PyTorch versions
!sed -i 's/from torch._six import inf/inf = float("inf")/' DynamicViT/utils.py

import sys
sys.path.append("DynamicViT")
sys.path.append("DynamicViT/models")

# Pretrained DynamicViT LV-ViT-M weights (keep ratio 0.7), from the official repository
!mkdir -p pretrained
!gdown -q 1dNab1B5ZOTVNpnpO6H1TsXKFM8BAlA3I -O pretrained/dynamic-vit_lv-m_r0.7.pth

In [ ]:
# Only on Colab, if the data are on Google Drive
from google.colab import drive
drive.mount('/content/drive')

## 2. Configuration

The dataset is not public (available on request, see README). Set `DATA_DIR`
to the folder containing the CSV files and the `DATASET/` image folder.

In [ ]:
import os

EXPERIMENT = "main"   # "main" | "heldout" | "sites"
SEED = 5              # 5, 16, 25

DATA_DIR = "/content/drive/MyDrive/data"     # <-- change to your data folder
IMG_DIR = os.path.join(DATA_DIR, "DATASET")
CSV_FILES = {
    "main":    {"train": "dataset.csv"},
    "heldout": {"train": "heldout_samples_train.csv", "test": "heldout_samples_test.csv"},
    "sites":   {"train": "heldout_sites_train.csv",   "test": "heldout_sites_test.csv"},
}
RUN_DIR = os.path.join(DATA_DIR, "runs", EXPERIMENT, f"seed{SEED}")
os.makedirs(RUN_DIR, exist_ok=True)

# Training settings used in the paper
NUM_EPOCHS = 25
LEARNING_RATE = 1e-4
PATIENCE = 8                      # early stopping on validation accuracy
BATCH_SIZE = 64 if EXPERIMENT == "main" else 32
T = 4                             # distillation temperature
LAMBDA_KL, LAMBDA_DISTILL, LAMBDA_RATIO = 0.5, 0.5, 2.0

# Settings that reproduce the paper (see the notes below)
USE_AUGMENTATION = False          # False = as in the paper
EVAL_CHECKPOINT = "final"         # "final" = as in the paper, or "best"
GENERALIZATION_EVAL_RESIZE = 256  # Resize before the 224 crop in the generalization tests

**Notes on reproducibility.**
- `USE_AUGMENTATION`: in the experiments reported in the paper, the training,
  validation and test subsets shared the same dataset object, so the evaluation
  transform (resize + center crop, no augmentation) was also applied during
  training. `False` reproduces this; `True` applies `train_transform`
  (random crop, flips, rotation, colour jitter, blur) to the training set only.
- `EVAL_CHECKPOINT`: the reported metrics were computed with the model weights
  at the end of training. `"best"` uses instead the epoch with the highest
  validation accuracy, which is also saved.

In [ ]:
import os
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import transforms
from timm.data.constants import IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD

from dylvvit import LVViTDiffPruning, LVViT_Teacher

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

## 3. Data

In [ ]:
class FabricsDataset(Dataset):
    """
    CSV with ';' separator and latin-1 encoding:
      column 0 -> image path relative to the image folder
      column 1 -> class label (1-20), shifted to 0-19
    """

    def __init__(self, csv_file, img_dir, transform=None):
        self.df = pd.read_csv(csv_file, delimiter=";", encoding="latin-1")
        self.img_dir = img_dir
        self.transform = transform

        label_col = self.df.columns[1]
        self.df.dropna(subset=[label_col], inplace=True)
        self.df[label_col] = self.df[label_col].astype(int)

        self.image_paths = self.df.iloc[:, 0].apply(
            lambda x: os.path.join(img_dir, x.replace("\\", "/")))
        self.labels = self.df.iloc[:, 1]

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.image_paths.iloc[idx]
        try:
            image = Image.open(img_path).convert("RGB")
            if self.transform:
                image = self.transform(image)
            return image, self.labels.iloc[idx] - 1
        except Exception as e:
            print(f"Error loading image {img_path}: {e}")
            return torch.zeros(3, 224, 224), -1   # removed by collate_fn


def collate_fn(batch):
    """Drop images that could not be loaded (label = -1)."""
    batch = [(x, y) for x, y in batch if y >= 0]
    if not batch:
        return torch.empty(0, 3, 224, 224), torch.empty(0, dtype=torch.int64)
    return torch.utils.data.default_collate(batch)


train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.1),
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=(5, 5))], p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_DEFAULT_MEAN, std=IMAGENET_DEFAULT_STD),
])


def eval_transform(resize=224):
    return transforms.Compose([
        transforms.Resize(resize),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_DEFAULT_MEAN, std=IMAGENET_DEFAULT_STD),
    ])


def split_70_15_15(n, seed):
    """
    70/15/15 split of the main experiment. As in the original notebook,
    random_split is called twice after setting the seed and the second call
    defines the split that was used.
    """
    set_seed(seed)
    lengths = [int(0.7 * n), int(0.15 * n)]
    lengths.append(n - sum(lengths))
    torch.utils.data.random_split(range(n), lengths)
    train, val, test = torch.utils.data.random_split(range(n), lengths)
    return train.indices, val.indices, test.indices

In [ ]:
train_csv = os.path.join(DATA_DIR, CSV_FILES[EXPERIMENT]["train"])

train_base = FabricsDataset(train_csv, IMG_DIR,
                            transform=train_transform if USE_AUGMENTATION else eval_transform())
eval_base = FabricsDataset(train_csv, IMG_DIR, transform=eval_transform())
n = len(train_base)

if EXPERIMENT == "main":
    train_idx, val_idx, test_idx = split_70_15_15(n, SEED)
else:
    # Generalization tests: the test set is a separate CSV, so the data are split 80/20 into train/val
    set_seed(SEED)
    train_size = int(0.8 * n)
    train_split, val_split = torch.utils.data.random_split(range(n), [train_size, n - train_size])
    train_idx, val_idx, test_idx = train_split.indices, val_split.indices, []

train_ds = Subset(train_base, train_idx)
val_ds = Subset(eval_base, val_idx)
test_ds = Subset(eval_base, test_idx)

loader_args = dict(batch_size=BATCH_SIZE, num_workers=2, pin_memory=True, collate_fn=collate_fn)
train_loader = DataLoader(train_ds, shuffle=True, **loader_args)
val_loader = DataLoader(val_ds, shuffle=False, **loader_args)
test_loader = DataLoader(test_ds, shuffle=False, **loader_args)

num_classes = train_base.labels.nunique()
print(f"Images: {n} | train {len(train_ds)} | val {len(val_ds)} | test {len(test_ds)} | classes {num_classes}")

## 4. Models

In [ ]:
LVVIT_M = dict(patch_size=16, embed_dim=512, depth=20, num_heads=8, mlp_ratio=3.,
               p_emb="4_2", skip_lam=2., return_dense=True)
PRUNING_LOC = [5, 10, 15]
TOKEN_RATIO = [0.7, 0.49, 0.343]


def build_student(num_classes, pretrained_path=None):
    """DynamicViT LV-ViT-M student. ImageNet-pretrained weights are loaded before replacing the heads."""
    model = LVViTDiffPruning(**LVVIT_M, mix_token=True, pruning_loc=PRUNING_LOC,
                             token_ratio=TOKEN_RATIO, viz_mode=True)
    if pretrained_path:
        model.load_state_dict(torch.load(pretrained_path, map_location="cpu")["model"])
    model.head = nn.Linear(model.head.in_features, num_classes)
    model.aux_head = nn.Linear(model.aux_head.in_features, num_classes)
    return model


PRETRAINED = "pretrained/dynamic-vit_lv-m_r0.7.pth"

set_seed(SEED)
model = build_student(num_classes, PRETRAINED).to(device)

**Teacher.** A dense LV-ViT-M (no token pruning) guides the student through
logit-level (KL) and token-level (MSE) distillation. As in the experiments
reported in the paper, its backbone is initialised from the same pretrained
checkpoint, its classification heads are re-initialised for the 20 classes,
and the whole teacher is kept frozen.

In [ ]:
teacher_model = LVViT_Teacher(**LVVIT_M)
teacher_model.load_state_dict(torch.load(PRETRAINED, map_location="cpu")["model"], strict=False)
teacher_model.head = nn.Linear(teacher_model.head.in_features, num_classes)
teacher_model.aux_head = nn.Linear(teacher_model.aux_head.in_features, num_classes)
teacher_model.to(device).eval()
for p in teacher_model.parameters():
    p.requires_grad = False

## 5. Training

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
from torch.cuda.amp import GradScaler, autocast
from tqdm import tqdm
import csv

# Balanced class weights (class imbalance)
labels_all = train_base.labels
class_weights = compute_class_weight("balanced", classes=np.unique(labels_all), y=labels_all)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

criterion_cls = nn.CrossEntropyLoss(weight=class_weights)
criterion_kl = nn.KLDivLoss(reduction="batchmean")
criterion_mse = nn.MSELoss()


def compute_ratio_loss(decision_masks, target_ratios):
    """Keeps the fraction of retained tokens close to the target ratio at each pruning stage."""
    loss = 0.0
    for mask, target in zip(decision_masks, target_ratios):
        actual_ratio = mask.mean(dim=1)
        loss += F.mse_loss(actual_ratio, torch.full_like(actual_ratio, target))
    return loss / len(decision_masks)


@torch.no_grad()
def accuracy_on(loader):
    model.eval()
    correct = total = 0
    for images, labels in loader:
        if images.numel() == 0:
            continue
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        outputs = outputs[0] if isinstance(outputs, tuple) else outputs
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return 100.0 * correct / total

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
scaler = GradScaler()

best_path = os.path.join(RUN_DIR, "best_dynamicvit.pth")
final_path = os.path.join(RUN_DIR, "DYNAMIC_VIT_model.pth")
log_path = os.path.join(RUN_DIR, "training_log.csv")
with open(log_path, "w", newline="") as f:
    csv.writer(f).writerow(["epoch", "train_loss", "train_acc", "val_acc"])

best_val_acc, epochs_no_improve = 0.0, 0

for epoch in range(NUM_EPOCHS):
    model.train()
    model.distill = True
    total_loss, correct, total, n_batches = 0.0, 0, 0, 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch + 1}/{NUM_EPOCHS}"):
        if images.numel() == 0:
            continue
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()

        with autocast():
            student_cls, student_aux, prev_decision, decision_masks = model(images)
            with torch.no_grad():
                teacher_cls, teacher_aux = teacher_model(images)

            # 1. Classification loss
            loss_cls = criterion_cls(student_cls, labels)
            # 2. Logit distillation (KL divergence)
            loss_kl = criterion_kl(F.log_softmax(student_cls / T, dim=1),
                                   F.softmax(teacher_cls / T, dim=1)) * (T * T)
            # 3. Token distillation on the tokens kept by the student
            token_mask = prev_decision.squeeze(-1).bool()
            loss_distill = criterion_mse(student_aux[token_mask], teacher_aux[token_mask])
            # 4. Token keep-ratio loss
            loss_ratio = compute_ratio_loss(decision_masks, TOKEN_RATIO)

            loss = (loss_cls + LAMBDA_KL * loss_kl
                    + LAMBDA_DISTILL * loss_distill + LAMBDA_RATIO * loss_ratio)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        n_batches += 1
        correct += (student_cls.argmax(1) == labels).sum().item()
        total += labels.size(0)

    train_loss, train_acc = total_loss / n_batches, 100.0 * correct / total
    val_acc = accuracy_on(val_loader)
    scheduler.step()
    print(f"Epoch {epoch + 1}: train loss {train_loss:.4f} | train acc {train_acc:.2f}% | val acc {val_acc:.2f}%")
    with open(log_path, "a", newline="") as f:
        csv.writer(f).writerow([epoch + 1, train_loss, train_acc, val_acc])

    if val_acc > best_val_acc:
        best_val_acc, epochs_no_improve = val_acc, 0
        torch.save(model.state_dict(), best_path)
    else:
        epochs_no_improve += 1
    if epochs_no_improve >= PATIENCE:
        print("Early stopping")
        break

torch.save(model.state_dict(), final_path)
print(f"Best val acc: {best_val_acc:.2f}% | weights saved in {RUN_DIR}")

## 6. Evaluation

Loads the checkpoint chosen in `EVAL_CHECKPOINT` (the final weights, as in the
paper). To evaluate an existing checkpoint without retraining, run sections
1–4 and then this section, after copying the file to `RUN_DIR`.

In [ ]:
import json
from sklearn.metrics import (classification_report, confusion_matrix, f1_score,
                             roc_auc_score)

eval_path = final_path if EVAL_CHECKPOINT == "final" else best_path
model.load_state_dict(torch.load(eval_path, map_location=device))
model.to(device).eval()
print("Evaluating", eval_path)


@torch.no_grad()
def predict(loader):
    probs, labels = [], []
    for images, y in loader:
        if images.numel() == 0:
            continue
        outputs = model(images.to(device))
        outputs = outputs[0] if isinstance(outputs, tuple) else outputs
        probs.append(torch.softmax(outputs, dim=1).cpu().numpy())
        labels.append(y.numpy())
    return np.concatenate(probs), np.concatenate(labels)


def macro_metrics(labels, preds, num_classes):
    """Accuracy and macro-averaged precision, recall and F1 (as in the paper)."""
    cm = np.zeros((num_classes, num_classes))
    for t, p in zip(labels, preds):
        cm[t, p] += 1
    tp = np.diag(cm)
    precision = tp / np.clip(cm.sum(0), 1, None)
    recall = tp / np.clip(cm.sum(1), 1, None)
    f1 = 2 * precision * recall / np.clip(precision + recall, 1e-9, None)
    return tp.sum() / cm.sum(), precision.mean(), recall.mean(), f1.mean()


def plot_confusion_matrix(labels, preds, num_classes, save_path):
    cm = confusion_matrix(labels, preds, labels=range(num_classes))
    names = [f"Class {i + 1}" for i in range(num_classes)]
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=False, fmt="d", cmap="ocean_r", cbar=True,
                xticklabels=names, yticklabels=names)
    threshold = cm.max() / 2
    for i in range(num_classes):
        for j in range(num_classes):
            plt.text(j + 0.5, i + 0.5, format(cm[i, j], "d"), ha="center", va="center",
                     color="white" if cm[i, j] > threshold else "black", fontsize=14)
    plt.xlabel("Predicted Label", fontsize=20, labelpad=10)
    plt.ylabel("True Label", fontsize=20, labelpad=10)
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    return cm

### 6a. Main experiment (Table 3, Fig. 2)
Run only with `EXPERIMENT = "main"`.

In [ ]:
assert EXPERIMENT == "main", "This cell is for the main experiment"

probs, labels = predict(test_loader)
preds = probs.argmax(1)

accuracy, precision, recall, f1 = macro_metrics(labels, preds, num_classes)
roc_auc = roc_auc_score(labels, probs, multi_class="ovr", average="weighted")

metrics = {"accuracy": accuracy, "precision": precision, "recall": recall,
           "f1": f1, "roc_auc": roc_auc}
metrics = {k: float(v) for k, v in metrics.items()}
for k, v in metrics.items():
    print(f"{k:10s} {100 * v:.2f}%")
with open(os.path.join(RUN_DIR, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)

# Fig. 2 (seed 5)
cm = plot_confusion_matrix(labels, preds, num_classes, os.path.join(RUN_DIR, "confusion_matrix.png"))

### 6b. Generalization tests (Table 4, Fig. 8 / Suppl. Table S2, Fig. S21)
Run only with `EXPERIMENT = "heldout"` or `"sites"`. The model is evaluated on
the separate test CSV of the held-out samples or sites.

In [ ]:
assert EXPERIMENT in ("heldout", "sites"), "This cell is for the generalization tests"

test_csv = os.path.join(DATA_DIR, CSV_FILES[EXPERIMENT]["test"])
gen_dataset = FabricsDataset(test_csv, IMG_DIR, transform=eval_transform(GENERALIZATION_EVAL_RESIZE))
gen_loader = DataLoader(gen_dataset, batch_size=64, shuffle=False, num_workers=2, collate_fn=collate_fn)

probs, labels = predict(gen_loader)
preds = probs.argmax(1)
present = np.unique(labels)

metrics = {
    "accuracy": float((labels == preds).mean()),
    "f1_macro": float(f1_score(labels, preds, average="macro", labels=present)),
    "f1_weighted": float(f1_score(labels, preds, average="weighted", labels=present)),
}
for k, v in metrics.items():
    print(f"{k:12s} {100 * v:.2f}%")
with open(os.path.join(RUN_DIR, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)

# Per-class precision, recall and F1 (%)
report = pd.DataFrame(classification_report(labels, preds, labels=present,
                                            output_dict=True, zero_division=0)).transpose()
report[["precision", "recall", "f1-score"]] *= 100
report.index = [f"Class {int(i) + 1}" if i.isdigit() else i for i in report.index]
report.to_csv(os.path.join(RUN_DIR, "per_class.csv"))
print(report.round(2).to_string())

# Fig. 8 (heldout, seed 5) / Fig. S21 (sites)
cm = plot_confusion_matrix(labels, preds, num_classes, os.path.join(RUN_DIR, "confusion_matrix.png"))

## 7. Mean ± std over the three seeds

After running the notebook with `SEED = 5`, `16` and `25` for the same
`EXPERIMENT`, this cell aggregates the saved metrics (sample standard
deviation, as in the paper).

In [ ]:
SEEDS = [5, 16, 25]
exp_dir = os.path.join(DATA_DIR, "runs", EXPERIMENT)

runs = [json.load(open(os.path.join(exp_dir, f"seed{s}", "metrics.json"))) for s in SEEDS]
print(f"{EXPERIMENT}: mean ± std over seeds {SEEDS}")
for key in runs[0]:
    values = 100 * np.array([r[key] for r in runs])
    print(f"  {key:12s} {values.mean():.2f} ± {values.std(ddof=1):.2f}")

if EXPERIMENT != "main":
    cols = ["precision", "recall", "f1-score"]
    tables = [pd.read_csv(os.path.join(exp_dir, f"seed{s}", "per_class.csv"), index_col=0)[cols]
              for s in SEEDS]
    stacked = np.stack([t.values for t in tables])
    mean = pd.DataFrame(stacked.mean(0), index=tables[0].index, columns=cols).round(2)
    std = pd.DataFrame(stacked.std(0, ddof=1), index=tables[0].index, columns=cols).round(2)
    summary = mean.astype(str) + "% ± " + std.astype(str) + "%"
    summary = summary.drop(index="accuracy", errors="ignore")
    summary.to_csv(os.path.join(exp_dir, "per_class_summary.csv"))
    print(summary.to_string())